# Thyra: end-to-end validation workflow (zero-installation)

This notebook reproduces the complete Thyra workflow from the *Nature Methods* Correspondence
**"Thyra: Bridging Mass Spectrometry Imaging and SpatialData for Unified Multi-Modal Analysis"** -
converting a raw MSI acquisition into the SpatialData standard, quality-checking the result, and
spatially querying a region of interest: entirely in the browser.

**How to run:** `Runtime > Run all`. Nothing is installed or configured on your machine. By
default the notebook runs on a small synthetic imzML dataset that Thyra generates on the fly
(about 30 MB, a few minutes end to end). Section 2 shows how to switch to the full manuscript
dataset from Zenodo, or to your own acquisition.

**Running locally instead of Colab:** any Jupyter environment works. Select a kernel with
Python 3.12 or 3.13 and run the same cells - the install cell installs Thyra into whichever
kernel is active. If you already have a Thyra environment (for example one created with
`uv sync` in the repository), select its interpreter as the kernel instead; the install cell
then sees Thyra present and changes nothing. A `ModuleNotFoundError: No module named 'thyra'`
in Section 3 means the active kernel never ran the install cell.

- Source code (MIT): https://github.com/M4i-Imaging-Mass-Spectrometry/thyra
- Documentation: https://M4i-Imaging-Mass-Spectrometry.github.io/thyra
- Example dataset: https://doi.org/10.5281/zenodo.18326569


## 1. Install Thyra

A single package brings the whole stack: Thyra plus a tested, version-pinned set of
scverse dependencies (spatialdata, anndata, zarr, dask).


In [ ]:
# "pandas<3" keeps the pandas that Colab preinstalls. The running kernel has
# numpy and pandas loaded already, so pip must not REPLACE either of them:
# an in-place swap crashes the Colab session (pandas) or breaks the half-
# loaded package on the next import (numpy).
%pip install -q thyra "pandas<3"

import thyra
print("Thyra", thyra.__version__)


## 2. Get an MSI dataset

imzML acquisitions come as a pair: the `.imzML` metadata file **and** the `.ibd` binary file.
Pick one of the three options below; the rest of the notebook is identical for all of them.


In [ ]:
# Option A (default): a small synthetic imzML dataset generated by Thyra itself
# (about 30 MB, a few seconds). This keeps "Run all" fast while exercising the
# exact same conversion pathway as real data.
!mkdir -p data
!thyra-example-data data/synthetic_brain.imzML


In [ ]:
# Option B: the full MALDI-MSI sagittal mouse brain dataset from the manuscript
# (public Zenodo record, 19.1 GB archive). Flip the flag to download it; expect
# a long download, and make sure the runtime has roughly 40 GB of free disk for
# the archive plus the extracted files.
DOWNLOAD_FULL_DATASET = False

if DOWNLOAD_FULL_DATASET:
    !wget -c -O data/MALDI-MSI_Sagittal_Mouse_Brain.tar.gz "https://zenodo.org/records/18326569/files/MALDI-MSI_Sagittal_Mouse_Brain.tar.gz?download=1"
    !tar -xzf data/MALDI-MSI_Sagittal_Mouse_Brain.tar.gz -C data/
    !find data -name "*.imzML" -o -name "*.ibd"


### Option C: bring your own data

Any imzML acquisition (the `.imzML` and `.ibd` pair together) goes through the same steps.
Two ways to get yours into Colab:

1. **Small files:** run the upload cell below and pick both files.
2. **Large files:** put them in your Google Drive and mount it, which avoids slow browser uploads.

Running locally, skip the upload cells and set the path directly, for example
`imzml = r'C:\data\my_acquisition.imzML'` - the conversion cell in Section 3 uses it as-is.

If conversion reports "Pixel size not found in metadata", pass `pixel_size_um=...` to
`convert_msi` in Section 3. Bruker `.d` and Waters `.raw` folders work too if you zip and
upload them the same way.


In [ ]:
# Option C1: upload your own .imzML + .ibd pair (small files)
# from google.colab import files
# up = files.upload()          # select BOTH the .imzML and the .ibd file
# imzml = [f for f in up if f.endswith('.imzML')][0]

# Option C2: mount Google Drive (large files)
# from google.colab import drive
# drive.mount('/content/drive')
# imzml = '/content/drive/MyDrive/path/to/your_data.imzML'


## 3. Convert MSI to SpatialData

Thyra auto-detects the input format, extracts the full spectra and instrument metadata,
harmonizes the mass axis with a physics-aware resampling strategy, and streams the data
into SpatialData elements (ion images, spectral matrix, pixel coordinates).


In [ ]:
import glob

# Uses the path set in Option C when given; otherwise picks the manuscript dataset
# when downloaded, falling back to the synthetic example from Option A.
if "imzml" not in globals():
    candidates = sorted(glob.glob("*.imzML") + glob.glob("data/**/*.imzML", recursive=True))
    imzml = next((p for p in candidates if "synthetic" not in p), candidates[0])
print("Converting:", imzml)

from thyra import convert_msi
ok = convert_msi(imzml, "output/thyra_example.zarr")
print("Conversion succeeded:", ok)


## 4. Load and inspect the SpatialData object


In [ ]:
import spatialdata as sd
sdata = sd.read_zarr("output/thyra_example.zarr")
print(sdata)
print(sdata.coordinate_systems)


## 5. Quality control

Because the output is a standard SpatialData/OME-NGFF object, QC is transparent and scriptable.


In [ ]:
import numpy as np

adata = sdata.tables[list(sdata.tables.keys())[0]]

tic = np.asarray(adata.X.sum(axis=1)).ravel()          # total ion current per pixel
print("TIC  median=%.3g  CV=%.2f%%" % (np.median(tic), 100 * tic.std() / tic.mean()))
low = tic < np.percentile(tic, 1)
print("Low-signal pixels:", int(low.sum()))


In [ ]:
# Visualize the TIC (total ion current) image inline
import matplotlib.pyplot as plt

img_key = list(sdata.images.keys())[0]
img = sdata.images[img_key]
arr = np.asarray(img.compute().data if hasattr(img, "compute") else img)
arr = np.squeeze(arr)
if arr.ndim == 3:  # (c, y, x) with several channels: show the first
    arr = arr[0]

plt.figure(figsize=(6, 5))
plt.imshow(arr, cmap="viridis")
plt.title(f"Ion image: {img_key}")
plt.axis("off")
plt.colorbar()
plt.show()


## 6. ROI query: co-localized molecular profile

Define a region of interest in the shared coordinate system and pull its mean spectrum
(its metabolic fingerprint). With aligned modalities in the same object, the identical
query returns cell-type composition or morphology.


In [ ]:
import numpy as np
import spatialdata as sd
from shapely.geometry import Polygon
from spatialdata import get_extent

# A rectangular ROI over the central half of the tissue, in the dataset's shared
# coordinate system (micrometers). Adjust the corners to target a specific structure.
cs = sdata.coordinate_systems[0]
extent = get_extent(sdata, coordinate_system=cs)
(xmin, xmax), (ymin, ymax) = extent["x"], extent["y"]
x0, x1 = xmin + 0.25 * (xmax - xmin), xmin + 0.75 * (xmax - xmin)
y0, y1 = ymin + 0.25 * (ymax - ymin), ymin + 0.75 * (ymax - ymin)

roi = Polygon([(x0, y0), (x1, y0), (x1, y1), (x0, y1)])
sub = sd.polygon_query(sdata, roi, target_coordinate_system=cs)

roi_table = sub.tables[list(sub.tables.keys())[0]]
print("Pixels in ROI:", roi_table.n_obs)
mean_spectrum = np.asarray(roi_table.X.mean(axis=0)).ravel()
top = np.argsort(mean_spectrum)[-10:][::-1]
print("Top m/z features in ROI:", roi_table.var_names[top].tolist())


## 7. Where to go next

- **Interactive, graphical inspection:** the converted `.zarr` store opens directly in **napari**
  (desktop) and **Vitessce** (browser): see the documentation for one-line loaders.
- **Downstream analysis:** the spectral table is a standard AnnData object, so Scanpy/Squidpy
  workflows apply directly (normalization, clustering, spatial statistics).
- **CLI without any environment management:** `uvx thyra input.imzML output.zarr` (or `pipx run thyra ...`).

*Questions or issues:* https://github.com/M4i-Imaging-Mass-Spectrometry/thyra/issues
